In [2]:
import api_key
from langchain_openai import ChatOpenAI
import os

index_name = "crime-manual"

In [3]:

import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [4]:
# 链接云端数据库

device='cuda' if torch.cuda.is_available() else 'cpu'

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)
# 使用 from_existing_index 直接连接已存在的云端索引 (仅读取)
vector_store = PineconeVectorStore.from_existing_index(
    embedding=embeddings,
    index_name="crime-manual-semantic"
)

In [14]:

# 配置mmr检索器
retriever = vector_store.as_retriever(
    search_type="mmr",  # 使用最大边际相关性（MMR）算法进行检索
    search_kwargs={"k": 12, "fetch_k": 50},
)
print("知识库连接成功！")

知识库连接成功！


In [6]:
# 配置deepseek模型
# === 单元格 3：配置 DeepSeek 大模型 ===
print("2. 初始化云端 DeepSeek 模型...")
llm = ChatOpenAI(
    model_name="deepseek-flash",
    openai_api_key=os.environ["DEEPSEEK_API_KEY"],
    openai_api_base="https://api.deepseek.com",
    max_tokens=2048,
    temperature=0.1,  # 调低温度以保证学术严谨性与防止幻觉
)
print("模型就绪！")

2. 初始化云端 DeepSeek 模型...
模型就绪！


In [7]:
test_response = llm.invoke("请只回复：连接正常")
print(test_response.content)

连接正常


In [ ]:
# 组装RAG链路，与之前的代码实现不同的这里先让llm理解用户的意图，转化为专业术语再去向量数据库中查询，这样子可以避免用户表达能力欠缺，导致检索器命中失败
print('构建问答链')

#进一步修改代码，这里启用记忆力机制
#导入所需要的库
from langchain_core.prompts import ChatMessagePromptTemplate,MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory



# # 重写提示词
# rewrite_prompt = PromptTemplate.from_template(
# """You are an expert FBI profiler and crime scene analyst. 
# Your task is to analyze the user's input and rewrite it into a highly precise, academic search string optimized for a vector database containing the "Crime Classification Manual".
# If the user asks broad concepts like "serial killers' behavior", you MUST expand and align the query with the actual taxonomy used in the manual, such as "organized, disorganized, mixed crime scenes, or primary intent/motivation".

#需要对重写提示词进行改写，支持历史对话
rewrite_prompt=ChatMessagePromptTemplate.from_message([
    ('system', 
    '''
    你是一名专业FBI犯罪侧写师和犯罪现场分析专家。
你的任务是：结合用户的历史对话上下文，深度分析用户的最新提问，并将其重写为一个极其精准、学术化的【纯英文检索词】，以便去向量数据库中检索《Crime Classification Manual》。

核心规则：
1. 必须进行指代消解。如果用户的最新提问中包含代词（如“那它有什么特征”、“第二种呢”），你必须根据历史对话找出具体指代的名词。
2. 必须对齐原书的专业术语本体（如 organized, disorganized, primary intent）。
3. 你的输出只能包含重写后的纯英文检索词！绝对不能包含任何中文、解释或对话前缀！'''),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human','{question}')
    ])
                                                      
                                                      
                                                

#User Query: {question}

# Output ONLY the rewritten English search query. Do not include any explanations, quotes, or conversational text."""
# )

# 构建查询重写链
query_rewrite = rewrite_prompt|llm|StrOutputParser()

#需要升级原函数，修改入参接收包含历史对话的字典
def retrieve_with_rewrite(chain_input:dict):
    smart_en_query=query_rewrite.invoke(chain_input).strip()
    print(f"\n[智能路由] 原始提问: {chain_input['question']}")
    print(f"[智能路由] 重写后的专业检索词: {smart_en_query}")
    
    return retriever.invoke(smart_en_query)

构建问答链


In [ ]:
# 组装最后的问答链
#加入历史记录占位符
qa_prompt = PromptTemplate.from_template("""
    你是一个专业文档问答助手，基于以下给出的背景信息回答用户的问题。
要求：
1. 使用中文回答；
2. 只依据资料内容，不要编造；
3. 如果资料中没有答案，请明确说“资料中没有找到相关信息”；
4. 必要时保留英文专业术语，并在后面给出中文解释。

背景资料：{context}
    
用户问题：{question}
    """)

enhanced_rag_chain = (
    {
        "context": RunnableLambda(retrieve_with_rewrite),
        "question": RunnablePassthrough(),
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

In [10]:
%pip install --upgrade --force-reinstall httpx brotli zstandard openai langchain-openai

Looking in indexes: https://pypi.org/simple/
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
  Using cached zstandard-0.25.0-cp313-cp313-win_amd64.whl.metadata (3.3 kB)
  Using cached openai-3.19.2-py3-none-any.whl.metadata (44 kB)
  Using cached langchain_openai-1.6.6-py3-none-any.whl.metadata (3.4 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached httpx2-2.13.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached jiter-0.17.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached typing_exte

ERROR: Could not install packages due to an OSError: [WinError 5] 拒绝访问。: 'd:\\anaconda\\lib\\site-packages\\_brotli.cp313-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



In [11]:
# 测试

user_query = "手册中对连环杀手的现场行为是如何分类的？"
print(f"\n开始处理查询：{user_query}")
response = enhanced_rag_chain.invoke(user_query)
print(f"\nAI 最终回答:\n{response}")


开始处理查询：手册中对连环杀手的现场行为是如何分类的？

[智能路由] 原始提问: 手册中对连环杀手的现场行为是如何分类的？
[智能路由] 重写后的专业检索词: Crime Classification Manual serial murder serial homicide crime scene behavior classification organized disorganized mixed crime scenes primary intent motivation taxonomy

AI 最终回答:
严格来说，提供的资料中没有出现“连环杀手（serial killer）”这一表述，也没有专门说明“连环杀手的现场行为”如何分类。因此，如果问题特指连环杀手，资料中没有找到相关信息。

不过，资料中有与杀人现场行为分类相关的内容，主要是性杀人（sexual homicide）和现场特征分类：

1. **CCM 的杀人动机分类**  
   《Crime Classification Manual》（CCM，犯罪分类手册）按犯罪人的主要意图将杀人分为 4 大类：  
   - criminal enterprise（犯罪牟利/犯罪企业）  
   - personal cause（个人原因）  
   - sexual intent（性意图）  
   - group cause（群体原因）

2. **性杀人（sexual homicide）的子类**  
   资料提到，sexual homicide category（性杀人类别）有 5 个子类，其中包括：  
   - organized crime scene murder（有组织型现场谋杀）  
   - disorganized（无组织型）  
   但资料未完整列出全部子类。

3. **现场行为可按 organized / disorganized / mixed 分类**  
   资料提到 “Sexual Homicide, Mixed”（混合型性杀人）：一个犯罪现场可能同时反映 organized（有组织的）和 disorganized（无组织的）特征。  
   其原因包括：  
   - 可能有多名犯罪人参与，因此表现出不同行为模式；  
   - 袭击可能开始时是有序、有计

In [16]:
user_query = "你详细讲解一下手册中'性杀人'的行为特征"
print(f"\n开始处理查询：{user_query}")
response = enhanced_rag_chain.invoke(user_query)
print(f"\nAI 最终回答:\n{response}")


开始处理查询：你详细讲解一下手册中性杀人的行为特征

[智能路由] 原始提问: 你详细讲解一下手册中性杀人的行为特征
[智能路由] 重写后的专业检索词: Crime Classification Manual sexual homicide behavioral characteristics taxonomy organized disorganized mixed crime scenes primary intent motivation victim selection modus operandi signature crime scene indicators postmortem sexual activity body disposal staging

AI 最终回答:
根据提供的《Crime Classification Manual》片段，资料中能直接归纳的性杀人（sexual homicide）行为特征如下。需要先说明：给定资料只是手册的若干片段，索引显示施虐型性杀人（Sexual Homicide, Sadistic）有犯罪现场指标、被害人学、伪装等章节，但正文细节未出现在资料中，因此以下不是手册对性杀人行为特征的完整总结。

### 一、混合型性杀人（Sexual Homicide, Mixed）的行为特征
资料指出，混合型性杀人的犯罪现场可能同时反映有组织型（organized）和无组织型（disorganized）特征，原因包括：

1. **可能有多名犯罪人参与**，因此会表现出不同的行为模式。
2. **攻击可能以有序、有计划的方式开始**，但因意外事件而恶化，例如无法控制被害人。
3. **主要动机可能只是强奸（rape）**，但被害人反抗或犯罪人的情绪状态导致行为升级（escalation），尤其见于敌意型或报复型强奸者（hostile or retaliatory type of rapist）。
4. **被害人选择可能体现有组织型犯罪人特征**：仔细选择并跟踪（stalking）被害人；但之后尸体未被藏匿或藏匿不佳。原文片段在此处截断。

### 二、有组织型性杀人（organized sexual homicide）的法医/行为发现
资料中“COMMON FORENSIC FINDINGS”部分提到，有组织型性杀人可